In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

In [ ]:


# 1.加载数据集
iris = load_iris()
X = iris.data   # 特征(4个连续特征)
y = iris.target # 标签，3个类别
feature_names = iris.feature_names
class_names = iris.target_names

print("特征名称：", feature_names)
print("类别名称：", class_names)
print("数据集总样本数：", X.shape[0])


In [ ]:

# 2.划分训练集、测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
# 3.构建高斯朴素贝叶斯模型，训练
gnb = GaussianNB()
gnb.fit(X_train, y_train)

# 4.预测 + 评估
y_pred = gnb.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\n测试集准确率：{acc:.2f}")

print("\n分类报告：")
print(classification_report(y_test, y_pred, target_names=class_names))

# 5.拿一条新样本预测
sample = [[5.1, 3.5, 1.4, 0.2]]
pred = gnb.predict(sample)
prob = gnb.predict_proba(sample)
print(f"\n新样本预测类别：{class_names[pred[0]]}")
print("各类别概率：", prob)


特征名称： ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
类别名称： ['setosa' 'versicolor' 'virginica']
数据集总样本数： 150

测试集准确率：0.98

分类报告：
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        19
  versicolor       1.00      0.92      0.96        13
   virginica       0.93      1.00      0.96        13

    accuracy                           0.98        45
   macro avg       0.98      0.97      0.97        45
weighted avg       0.98      0.98      0.98        45


新样本预测类别：setosa
各类别概率： [[1.00000000e+00 7.82732978e-17 1.66528708e-24]]


## 朴素贝叶斯


# 朴素贝叶斯中「朴素」是什么
> **朴素 = 特征条件独立假设（naive）**
简单一句话：**朴素，就是天真地假设所有特征之间相互独立，互不影响。**

## 完整解释
贝叶斯公式基础：
$$P(Y|X)=\frac{P(X|Y)P(Y)}{P(X)}$$
$X$ 是多个特征 $X_1,X_2,...,X_n$。

想要算 $P(X_1,X_2,...,X_n|Y)$，真实情况下特征之间大多是有关联的（比如身高和体重相关），联合概率很难估计。

**朴素贝叶斯做了一个很强的简化假设：**
> 在给定类别 $Y$ 的条件下，各个特征 $X_1,X_2...X_n$ 相互独立。

于是联合概率可以拆成乘积：
$$P(X_1,X_2,...,X_n|Y)=\prod_{i=1}^n P(X_i|Y)$$

这个假设**现实里几乎很少严格成立**，只是为了简化计算，所以叫“朴素/天真（naive）”。

## 举个例子
垃圾邮件分类：
特征：包含词A、词B、词C。
朴素假设：**如果这是垃圾邮件，词A出现与否，不会影响词B、词C是否出现**。
真实情况：词语之间明显有关联，这个假设是强行简化。

## 特点
✅ 优点：计算简单、训练快，小数据集也能用，适合文本分类
❌ 缺点：**特征独立假设经常不成立**，有时候会损失精度

## 一句话总结考点（应付考试）
> “朴素”指**条件独立性假设**：给定类别时，各个特征相互条件独立。

# 朴素贝叶斯三种模型对比
> 核心区别：**对特征 $P(X_i|Y)$ 的概率分布假设不一样**
> 三者都保留「朴素」的条件独立假设，只是特征类型不同，选不同分布。

## 1. 多项式朴素贝叶斯 MultinomialNB
### 适用数据：**离散计数特征**
- 典型场景：文本分类（词袋，单词出现次数）
- 分布假设：特征服从**多项式分布**，描述事件发生次数
- 输入要求：特征是非负整数（词频 count）
- 原理：计算每个类别下，每个词出现的概率
- 细节：自带平滑（拉普拉斯平滑），解决训练集里某些词概率为0的问题
> 例：邮件里“中奖”出现2次，“转账”出现1次，用它做垃圾邮件分类。

## 2. 伯努利朴素贝叶斯 BernoulliNB
### 适用数据：**二值0-1特征**
- 特征只能是0或1：代表**有没有这个特征，不关心出现多少次**
- 分布假设：伯努利分布（0/1单次试验）
- 典型场景：文本分类（词是否存在，不是词频）
> 例：邮件是否包含“中奖”，出现1次和出现100次都记为1。
> 区别Multinomial：Multinomial看次数；Bernoulli只看有无。

## 3. 高斯朴素贝叶斯 GaussianNB
### 适用数据：**连续数值特征**
- 分布假设：连续特征服从**高斯（正态）分布**
- 思路：训练时对每个类别，计算该类别下每个特征的均值、方差；预测时代入正态概率密度公式算出 $P(X_i|Y)$
- 典型场景：鸢尾花数据集（花瓣长度、宽度这类连续实数）
> 例：身高、体重、温度这类连续变量。

# 快速对比表
|模型|特征类型|分布|经典例子|
| ---- | ---- | ---- | ---- |
|GaussianNB|连续实数|高斯正态分布|鸢尾花，身高体重|
|MultinomialNB|离散计数（≥0整数）|多项式分布|词袋词频文本分类|
|BernoulliNB|二值0/1|伯努利分布|词是否在文档中出现|

# 考试简答版
1. **高斯朴素贝叶斯**：假设连续特征服从高斯分布，用于连续型特征。
2. **多项式朴素贝叶斯**：假设离散计数特征服从多项式分布，多用于文本词频。
3. **伯努利朴素贝叶斯**：假设特征为0-1二值，服从伯努利分布，只判断特征是否存在。

## 补充考点：拉普拉斯平滑
多项式/伯努利朴素贝叶斯会用到。
问题：训练集中某个词在某类别下一次都没出现，$P(X_i|Y)=0$，乘积直接为0，预测失效。
拉普拉斯平滑：分子+1，分母+类别特征总数，**避免概率为0**。高斯朴素贝叶斯一般不需要。

## 易混点
MultinomialNB：统计次数；BernoulliNB：只看有无。
比如一句话：“中奖 中奖”
- Multinomial：“中奖”计数=2
- Bernoulli：“中奖”=1（存在）

要不要我给你写一段极简sklearn代码，分别跑这三个模型，直观感受输入数据的差异？